In [ ]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset

from transformers import (
    set_seed,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments, 
    Trainer,
)
from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import torch
from datasets import load_dataset
def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)
import torch.nn.functional as F


device = 'cuda' if torch.cuda.is_available() else 'cpu'


# 1 LLMs

Selecting batch size

In [ ]:
batch_size = 64

Zero-shot prompt tempalte for 2 label task: Hate + Neutral

In [7]:
prompt_template = '''You are tasked with annotating speech. Your response must be a single valid number:
1 for Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech,
2 for Normal Speech.

Provide only the number corresponding to the category. Do not include any explanation or additional text.
Do you think the following comment is Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech, speech or Normal speech?
\n"{comment}"\n
Your Answer:
'''



A list of fine-tuned models is available on Hugging Face

In [8]:
models_list_HF = {"Llama3.2-1B": {
                    "Base": "unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit",
                    "Human": "anonymousOWSHateLLM/Hate-Llama3.2-1B.human.2_label",
                    "Lgb": "anonymousOWSHateLLM/Hate-Llama3.2-1B.Lgb.2_label",
                    "Mean": "anonymousOWSHateLLM/Hate-Llama3.2-1B.Mean.2_label",
                    "Vote": "anonymousOWSHateLLM/Hate-Llama3.2-1B.Vote.2_label",
                    "Human-Lgb": "anonymousOWSHateLLM/Hate-Llama3.2-1B.Human_Lgb.2_label",
                    },
                "Qwen2.5-14B": {
                    "Base": "unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
                    "Human": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Human.2_label",
                    "Lgb": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Lgb.2_label",
                    "Mean": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Mean.2_label",
                    "Vote": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Vote.2_label",
                    "Human-Lgb": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Human_Lgb.2_label",
                    }}


Evaluation set

In [9]:
dataset = load_dataset("anonymousOWSHateLLM/df_eval_16")

README.md:   0%|          | 0.00/555 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.17M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/31365 [00:00<?, ? examples/s]

In [50]:
train_set = load_dataset("anonymousOWSHateLLM/train_set")

README.md:   0%|          | 0.00/383 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/126475 [00:00<?, ? examples/s]

In [11]:
df_val = pd.DataFrame(dataset['train'])

In [12]:
df_val.ds.value_counts()

ds
HateXplain      3846
GermEval18      3532
AHSD            3000
ViHSD           2672
Sexism          2632
GermEval19      2507
Gahd            2198
GermEval21      2085
Chileno         1928
HateEval-spa    1286
Haternet        1205
US_election     1117
HateEval-eng    1000
Covid            971
AbusEval         860
HASOC            526
Name: count, dtype: int64

In [13]:
print(df_val.groupby(['ds', 'label_id']).size().unstack(fill_value=0))

label_id         1     2
ds                      
AHSD          2494   506
AbusEval       178   682
Chileno        110  1818
Covid          190   781
Gahd           939  1259
GermEval18    1202  2330
GermEval19     840  1667
GermEval21     769  1316
HASOC          134   392
HateEval-eng   427   573
HateEval-spa   556   730
HateXplain    2283  1563
Haternet       305   900
Sexism         350  2282
US_election    140   977
ViHSD          482  2190


Selecting Group model Llama3.2-1B or Qwen2.5-14B 
Test set: 1 or 2

In [14]:
model_group = "Llama3.2-1B"

In [16]:
def process_task(texts, model, tokenizer, stop_token_id):
    encoding = tokenizer(texts, padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits  # Shape: [batch_size, sequence_length, vocab_size]
    last_token_logits = logits[:, -1, :]  # Shape: [vocab_size]
    probabilities = torch.softmax(last_token_logits, dim=-1)
    indices = torch.tensor(stop_token_id)
    selected_probs_1 = probabilities[:, indices[0]].float().cpu().numpy()
    selected_probs_2 = probabilities[:, indices[1]].float().cpu().numpy()
    return selected_probs_1, selected_probs_2

In [15]:
def preprocess(text, model_id, tokenizer):
    user_message_content = prompt_template.format(comment=text)
    user_message = {
        "role": "user",
        "content": user_message_content
    }

    if "Qwen" in model_id:
        system_message =  {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant"}
    else:
        system_message =  {"role": "system", "content": "You are a helpful assistant"}
    messages = [system_message, user_message]
    messages = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    messages = messages


    return messages


In [21]:
def run_eval():
    model_probs_dict = {}

    model_list = models_list_HF[model_group]
    df_eval = df_val


    for key, model_id in model_list.items():

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=500,
            dtype=getattr(torch, "bfloat16"),
        )
        FastLanguageModel.for_inference(model)
        tokenizer.padding_side = "left"

        stop_token_id = [16, 17]

        if model_group == "Qwen2.5-14B":
            stop_token_id = tokenizer(["12"])['input_ids'][0]
        elif model_group == "Llama3.2-1B":
            stop_token_id = [16, 17]


        df_eval["prompt"] = df_eval["text"].apply(lambda text: preprocess(text, model_id, tokenizer))


        texts = []
        probs_label_1 = []
        probs_label_2 = []

        prompts = df_eval['prompt'].tolist()

        for i in tqdm(range(0, len(prompts), batch_size)):
            batch = prompts[i:i+batch_size]
            selected_probs_1, selected_probs_2 = process_task(batch, model, tokenizer, stop_token_id)
            probs_label_1.extend(selected_probs_1.tolist())
            probs_label_2.extend(selected_probs_2.tolist())
            torch.cuda.empty_cache()
            torch.cuda.synchronize()


        model_probs_dict[key] = {
            "probs_label_1": probs_label_1,
            "probs_label_2": probs_label_2,
        }
    return model_probs_dict

def make_report(model_probs_dict, df_eval):
    report = {}
    for ds in df_eval['ds'].unique():
        report[ds] = {}
    report["Mean"] = {}

    y_true = np.array(df_eval['label_id'] == 1, dtype=int)
    df_eval['y_ture'] = y_true

    for model, probs in model_probs_dict.items():
        y_prob_label_1 = probs["probs_label_1"]
        df_eval['y_pred'] = y_prob_label_1 > np.mean(y_prob_label_1)
        df_eval['probs'] = y_prob_label_1

        report["Mean"][model] = {
        "acc": round(accuracy_score(y_true, df_eval['y_pred'])* 100, 1), 
        "f1": round(f1_score(y_true,df_eval['y_pred'], average='macro')* 100, 1)
            }
        
        for ds in df_eval['ds'].unique():
            tmp_df = df_eval.loc[df_eval['ds'] == ds]

            y_pred = tmp_df['y_pred'] 


            acc = round(accuracy_score(tmp_df['y_ture'],y_pred)* 100, 1) 
            f1 = round(f1_score(tmp_df['y_ture'], y_pred, average='macro')* 100, 1) 
            report[ds][model] = {
                            "acc": acc, 
                            "f1": f1
                            }
    
    report_df = pd.DataFrame(report).T
    return report_df

In [ ]:
model_group = "Llama3.2-1B"
result = {}
probs = run_eval()


In [19]:
import pickle
with open("Llama1B_probs.pkl", "rb") as f:
    probs = pickle.load(f)

In [22]:
report = make_report(probs, df_val)

In [23]:
report

,Base,Human,Lgb,Mean,Vote,Human-Lgb
HateXplain,"{'acc': 53.6, 'f1': 53.3}","{'acc': 58.4, 'f1': 58.4}","{'acc': 67.7, 'f1': 67.0}","{'acc': 60.8, 'f1': 60.7}","{'acc': 61.3, 'f1': 61.3}","{'acc': 70.4, 'f1': 67.3}"
GermEval18,"{'acc': 42.5, 'f1': 40.0}","{'acc': 44.7, 'f1': 42.8}","{'acc': 68.6, 'f1': 67.2}","{'acc': 55.0, 'f1': 54.9}","{'acc': 53.5, 'f1': 53.3}","{'acc': 58.8, 'f1': 58.7}"
Haternet,"{'acc': 35.3, 'f1': 34.9}","{'acc': 44.2, 'f1': 44.0}","{'acc': 52.3, 'f1': 49.8}","{'acc': 38.6, 'f1': 38.5}","{'acc': 36.3, 'f1': 35.9}","{'acc': 50.9, 'f1': 50.5}"
AHSD,"{'acc': 44.0, 'f1': 40.8}","{'acc': 41.5, 'f1': 40.5}","{'acc': 51.7, 'f1': 49.7}","{'acc': 44.4, 'f1': 43.6}","{'acc': 45.5, 'f1': 44.5}","{'acc': 73.6, 'f1': 66.3}"
AbusEval,"{'acc': 64.5, 'f1': 56.2}","{'acc': 75.9, 'f1': 52.7}","{'acc': 80.3, 'f1': 66.1}","{'acc': 77.7, 'f1': 51.5}","{'acc': 77.7, 'f1': 52.2}","{'acc': 78.7, 'f1': 61.7}"
GermEval19,"{'acc': 42.8, 'f1': 40.8}","{'acc': 44.8, 'f1': 43.5}","{'acc': 67.9, 'f1': 66.8}","{'acc': 55.4, 'f1': 55.4}","{'acc': 53.3, 'f1': 53.1}","{'acc': 59.2, 'f1': 59.1}"
ViHSD,"{'acc': 50.8, 'f1': 47.1}","{'acc': 75.0, 'f1': 59.3}","{'acc': 64.6, 'f1': 58.3}","{'acc': 65.9, 'f1': 58.5}","{'acc': 64.5, 'f1': 57.6}","{'acc': 76.3, 'f1': 66.1}"
HateEval-eng,"{'acc': 49.5, 'f1': 49.5}","{'acc': 58.2, 'f1': 51.5}","{'acc': 64.0, 'f1': 62.1}","{'acc': 61.9, 'f1': 56.8}","{'acc': 62.1, 'f1': 57.6}","{'acc': 67.2, 'f1': 66.2}"
HateEval-spa,"{'acc': 42.6, 'f1': 38.8}","{'acc': 53.7, 'f1': 53.7}","{'acc': 49.9, 'f1': 49.8}","{'acc': 46.3, 'f1': 42.9}","{'acc': 47.2, 'f1': 43.3}","{'acc': 54.4, 'f1': 53.9}"
Gahd,"{'acc': 54.9, 'f1': 54.7}","{'acc': 45.1, 'f1': 37.8}","{'acc': 61.4, 'f1': 61.3}","{'acc': 56.9, 'f1': 55.3}","{'acc': 57.3, 'f1': 55.7}","{'acc': 57.9, 'f1': 56.8}"


Qwen2.5-14B

In [ ]:
model_group = "Qwen2.5-14B"

result = {}
probs = run_eval()


In [ ]:
with open("Qwen14_probs.pkl", "rb") as f:
    probs = pickle.load(f)

In [30]:
report = make_report(probs, df_val)

In [31]:
report

,Human,Human-Lgb,Base,Lgb,Mean,Vote
HateXplain,"{'acc': 73.9, 'f1': 70.0}","{'acc': 75.6, 'f1': 73.1}","{'acc': 66.3, 'f1': 54.6}","{'acc': 73.2, 'f1': 69.5}","{'acc': 74.9, 'f1': 72.2}","{'acc': 73.8, 'f1': 72.4}"
GermEval18,"{'acc': 82.8, 'f1': 80.9}","{'acc': 83.2, 'f1': 80.5}","{'acc': 75.8, 'f1': 75.3}","{'acc': 82.6, 'f1': 79.9}","{'acc': 82.6, 'f1': 79.5}","{'acc': 80.1, 'f1': 75.0}"
Haternet,"{'acc': 69.5, 'f1': 67.3}","{'acc': 77.0, 'f1': 73.5}","{'acc': 46.4, 'f1': 46.3}","{'acc': 75.0, 'f1': 72.2}","{'acc': 77.4, 'f1': 74.0}","{'acc': 81.2, 'f1': 76.0}"
AHSD,"{'acc': 81.9, 'f1': 74.9}","{'acc': 79.5, 'f1': 72.9}","{'acc': 91.9, 'f1': 84.9}","{'acc': 72.8, 'f1': 67.1}","{'acc': 81.5, 'f1': 74.8}","{'acc': 63.8, 'f1': 59.6}"
AbusEval,"{'acc': 82.1, 'f1': 68.7}","{'acc': 82.1, 'f1': 66.2}","{'acc': 66.9, 'f1': 63.4}","{'acc': 80.5, 'f1': 67.1}","{'acc': 81.5, 'f1': 68.2}","{'acc': 81.4, 'f1': 67.2}"
GermEval19,"{'acc': 81.3, 'f1': 78.9}","{'acc': 82.2, 'f1': 79.1}","{'acc': 72.2, 'f1': 71.8}","{'acc': 81.2, 'f1': 78.3}","{'acc': 81.9, 'f1': 79.0}","{'acc': 79.5, 'f1': 74.0}"
ViHSD,"{'acc': 85.0, 'f1': 76.0}","{'acc': 86.9, 'f1': 77.5}","{'acc': 78.7, 'f1': 71.9}","{'acc': 87.3, 'f1': 75.9}","{'acc': 87.1, 'f1': 76.0}","{'acc': 86.4, 'f1': 70.9}"
HateEval-eng,"{'acc': 67.2, 'f1': 66.9}","{'acc': 70.0, 'f1': 69.4}","{'acc': 63.0, 'f1': 61.7}","{'acc': 70.0, 'f1': 69.8}","{'acc': 69.7, 'f1': 69.5}","{'acc': 69.0, 'f1': 68.3}"
HateEval-spa,"{'acc': 65.2, 'f1': 64.6}","{'acc': 67.5, 'f1': 67.4}","{'acc': 57.9, 'f1': 54.6}","{'acc': 67.2, 'f1': 67.0}","{'acc': 65.2, 'f1': 65.1}","{'acc': 67.8, 'f1': 67.8}"
Gahd,"{'acc': 77.7, 'f1': 77.7}","{'acc': 77.3, 'f1': 77.0}","{'acc': 73.5, 'f1': 73.5}","{'acc': 75.5, 'f1': 75.0}","{'acc': 75.8, 'f1': 75.0}","{'acc': 71.6, 'f1': 69.3}"


# 2 Bert

In [ ]:

def classify_texts_Bert(texts=None, batch_size=128, model=None, tokenizer=None, return_probs=False):
    predictions = []
    probs_all = []
    model = model.to("cuda")
    model.eval()
    print()

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        encoded = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True, max_length=512)
        encoded = {k: v.to("cuda") for k, v in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1) 
            batch_preds = torch.argmax(probs, dim=1).tolist()
            batch_probs = probs.tolist()
            # batch_preds = torch.argmax(logits, dim=1).tolist()
            predictions.extend(batch_preds)
            probs_all.extend(batch_probs)
    probs_all = np.vstack(probs_all)
    if return_probs:
        return probs_all
    return predictions

In [ ]:
def finetuneBert(
        df=None,
        base=None,
        tokenizer=None,
        save_path=None,
):
    
    df = df.sample(frac=1, random_state=def_seed).reset_index(drop=True)

    dataset = Dataset.from_pandas(df)

    def tokenize_and_format(batch):
        encodings = tokenizer(batch["text"] )
        encodings["len"] = [len(ids) for ids in encodings["input_ids"]]
        encodings["labels"] = batch["labels"] 
        return encodings


    tokenized_dataset = dataset.map(
        tokenize_and_format,
        batched = True,
        batch_size = 512,
        num_proc=10
    )

    filtered_dataset = tokenized_dataset.filter(
        lambda example: example['len'] < 512
    )

    training_args = TrainingArguments(
        output_dir=save_path,
        overwrite_output_dir=True,
        # num_train_epochs=train_conf["num_train_epochs"],
        num_train_epochs=3,
        # max_steps=2000,
        save_strategy="steps",        
        save_steps=2000,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=1,
        learning_rate=1e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.2,
        max_grad_norm=0.2,
        logging_steps=200,
        fp16=False,
        bf16=True,
        gradient_accumulation_steps=2,
        gradient_checkpointing=True,
        report_to="none",              
        eval_steps=5000,
        max_steps=-1,                  
        log_level="debug",
        dataloader_num_workers=2,
    )


    trainer = Trainer(
        model=base,
        args=training_args,
        train_dataset=filtered_dataset,
        tokenizer=tokenizer,
    )

    trainer.train()
    return trainer
    

Labeling for Ows4L Bert, has been trained with all 16 dataset

In [54]:
df_train = pd.DataFrame(train_set['train'])

In [55]:
df_train.ds.value_counts()

ds
AHSD            21783
HateXplain      15299
AbusEval        13240
Sexism          10904
GermEval19       9698
HateEval-eng     9000
Gahd             8797
ViHSD            8061
Chileno          7572
HateEval-spa     5309
GermEval18       5009
Haternet         4794
HASOC            2373
GermEval21       2071
US_election      1283
Covid            1282
Name: count, dtype: int64

In [56]:
df_train.iloc[0]

text        @dualipuhs Leave my ship alone you whore https...
language                                                  eng
ds                                               HateEval-eng
label_id                                                    1
Name: 0, dtype: object

In [ ]:
Ows4L = "anonymousOWSHateLLM/Ows4L_16"
OwsSpa = "anonymousOWSHateLLM/OwsSpa"
OwsDeu = "anonymousOWSHateLLM/OwsDeu"
OwsEng = "anonymousOWSHateLLM/OwsEng"

HateBert = "GroNLP/hateBERT"
Bert = "google-bert/bert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(Ows4L, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(Ows4L)

In [58]:
df_train['labels'] = (df_train['label_id'] == 1).astype(int)
df_train['labels'].value_counts()

labels
0    75612
1    50863
Name: count, dtype: int64

In [ ]:
trainer = finetuneBert(
    df_train,
    model,
    tokenizer,
    "Ows4L_16"
)

In [ ]:
output_pred = classify_texts_Bert(
    texts=df_val['text'].fillna('').tolist(),
    model=model,
    tokenizer=tokenizer
)

In [ ]:
model_probs_dict = {}
model_probs_dict['16_set'] = {}
model_probs_dict['16_set']['Ows4L'] = output_pred

In [46]:
with open("Bert_report.pkl", "rb") as f:
    model_probs_dict = pickle.load(f)

In [47]:
df_eval = df_val
report = {}
for ds in df_eval['ds'].unique():
    report[ds] = {}
report["Mean"] = {}

y_true = np.array(df_eval['label_id'] == 1, dtype=int)
df_eval['y_ture'] = y_true

for test_type, model_list in model_probs_dict.items():
        for model_id, y_pred in model_list.items():
            df_eval['y_pred'] = y_pred
            report["Mean"][test_type + "_" + model_id] = {
            "f1": round(f1_score(y_true,y_pred)* 100, 1)
                }
            
            for ds in df_eval['ds'].unique():
                tmp_df = df_eval.loc[df_eval['ds'] == ds]
                y_pred = tmp_df['y_pred']

                f1 = round(f1_score(tmp_df['y_ture'], y_pred, average='macro')* 100, 1) 
                report[ds][test_type + "_" + model_id] = {
                                "f1": f1
                                }

In [49]:
report_df = pd.DataFrame(report).T
report_df

,spa_OWS_spa,spa_Bert,spa_Hate,deu_OWS_deu,deu_Bert,deu_Hate,eng_OWS_spa,eng_Bert,eng_Hate,7set_OWS_spa,7set_Bert,7set_Hate,16set_OWS_spa,16set_Bert,16set_Hate,7fake_OWS_spa,7fake_Bert,7fake_Hate
HateXplain,{'f1': 28.9},{'f1': 30.3},{'f1': 54.7},{'f1': 59.7},{'f1': 52.6},{'f1': 55.5},{'f1': 77.9},{'f1': 77.7},{'f1': 77.5},{'f1': 77.6},{'f1': 77.7},{'f1': 77.6},{'f1': 77.5},{'f1': 78.0},{'f1': 77.1},{'f1': 78.4},{'f1': 78.9},{'f1': 77.8}
GermEval18,{'f1': 39.7},{'f1': 40.1},{'f1': 39.8},{'f1': 81.4},{'f1': 76.4},{'f1': 72.8},{'f1': 41.0},{'f1': 42.0},{'f1': 42.4},{'f1': 83.1},{'f1': 80.5},{'f1': 75.5},{'f1': 82.6},{'f1': 80.5},{'f1': 75.7},{'f1': 78.6},{'f1': 76.3},{'f1': 74.6}
Haternet,{'f1': 68.1},{'f1': 67.3},{'f1': 64.9},{'f1': 56.8},{'f1': 45.7},{'f1': 48.7},{'f1': 44.4},{'f1': 43.0},{'f1': 45.1},{'f1': 48.0},{'f1': 45.2},{'f1': 46.7},{'f1': 71.9},{'f1': 69.3},{'f1': 67.7},{'f1': 49.1},{'f1': 47.2},{'f1': 45.2}
AHSD,{'f1': 14.4},{'f1': 16.1},{'f1': 39.5},{'f1': 45.3},{'f1': 28.8},{'f1': 48.5},{'f1': 91.7},{'f1': 91.7},{'f1': 90.6},{'f1': 54.0},{'f1': 49.4},{'f1': 50.6},{'f1': 91.9},{'f1': 91.4},{'f1': 91.0},{'f1': 79.4},{'f1': 78.4},{'f1': 79.1}
AbusEval,{'f1': 44.2},{'f1': 44.2},{'f1': 53.5},{'f1': 51.4},{'f1': 48.6},{'f1': 58.6},{'f1': 70.7},{'f1': 67.0},{'f1': 67.1},{'f1': 52.5},{'f1': 52.1},{'f1': 55.1},{'f1': 72.1},{'f1': 68.6},{'f1': 67.1},{'f1': 55.9},{'f1': 51.7},{'f1': 56.7}
GermEval19,{'f1': 39.9},{'f1': 40.0},{'f1': 39.9},{'f1': 78.8},{'f1': 74.8},{'f1': 70.8},{'f1': 41.3},{'f1': 41.7},{'f1': 41.7},{'f1': 74.6},{'f1': 73.4},{'f1': 69.8},{'f1': 79.1},{'f1': 77.5},{'f1': 73.4},{'f1': 73.5},{'f1': 72.0},{'f1': 71.0}
ViHSD,{'f1': 45.0},{'f1': 45.0},{'f1': 45.7},{'f1': 55.5},{'f1': 49.1},{'f1': 50.9},{'f1': 45.0},{'f1': 45.2},{'f1': 45.6},{'f1': 74.0},{'f1': 72.2},{'f1': 67.4},{'f1': 73.8},{'f1': 72.3},{'f1': 67.9},{'f1': 72.8},{'f1': 71.0},{'f1': 64.7}
HateEval-eng,{'f1': 36.4},{'f1': 37.4},{'f1': 54.0},{'f1': 58.8},{'f1': 47.2},{'f1': 56.4},{'f1': 75.9},{'f1': 77.3},{'f1': 75.6},{'f1': 60.0},{'f1': 58.7},{'f1': 60.5},{'f1': 76.2},{'f1': 77.0},{'f1': 76.3},{'f1': 61.4},{'f1': 60.2},{'f1': 62.2}
HateEval-spa,{'f1': 76.6},{'f1': 75.9},{'f1': 73.5},{'f1': 46.9},{'f1': 38.3},{'f1': 38.4},{'f1': 36.3},{'f1': 36.2},{'f1': 36.7},{'f1': 38.6},{'f1': 36.9},{'f1': 36.7},{'f1': 79.0},{'f1': 78.6},{'f1': 76.7},{'f1': 38.2},{'f1': 38.5},{'f1': 36.5}
Gahd,{'f1': 36.9},{'f1': 37.7},{'f1': 36.5},{'f1': 73.2},{'f1': 71.0},{'f1': 70.0},{'f1': 38.9},{'f1': 39.2},{'f1': 40.8},{'f1': 55.5},{'f1': 53.2},{'f1': 53.9},{'f1': 73.2},{'f1': 72.9},{'f1': 71.2},{'f1': 45.9},{'f1': 45.7},{'f1': 46.5}
